In [157]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
)

In [72]:
root_dir = Path.cwd().parent.parent

combined_path = root_dir / "data" / "processed" / "combined_sample.parquet"
flow_path = root_dir / "data" / "processed" / "flow_training_sample.parquet"

In [73]:
combined_features = pd.read_parquet(combined_path)
flow_features = pd.read_parquet(flow_path)

In [74]:
print(f"Combined feature Shape: {combined_features.shape}")
print(f"Flow feature Shape: {flow_features.shape}")

Combined feature Shape: (204196, 221)
Flow feature Shape: (204196, 85)


In [75]:
combined_features.head()

,pkt_stream,pkt_src_mac,pkt_dst_mac,pkt_src_ip,pkt_dst_ip,pkt_src_port,pkt_dst_port,pkt_inter_arrival_time,pkt_time_since_previously_displayed_frame,pkt_port_class_dst,...,flow_Active Mean,flow_Active Std,flow_Active Max,flow_Active Min,flow_Idle Mean,flow_Idle Std,flow_Idle Max,flow_Idle Min,flow_Label,flow_matched
0,185,3c:18:a0:41:c3:a0,Arlo Q Indoor Camera,63.34.92.22,192.168.137.175,443,37446,282.540570,0.000001,2,...,0.000,0.00000,0.0,0.0,0.0,0.00,0.0,0.0,NeedManualLabel,True
1,949,3c:18:a0:41:c3:a0,Arlo Q Indoor Camera,52.209.241.1,192.168.137.175,443,45428,2270.156822,0.000001,2,...,0.000,0.00000,0.0,0.0,0.0,0.00,0.0,0.0,NeedManualLabel,True
2,572,Amazon Echo Show,3c:18:a0:41:c3:a0,192.168.137.172,54.167.177.211,50716,443,1769.096959,0.002118,1,...,234549.625,291474.28125,5328027.0,0.0,26881768.0,3295999.75,29989216.0,0.0,NeedManualLabel,True
3,721,3c:18:a0:41:c3:a0,Arlo Q Indoor Camera,54.77.81.52,192.168.137.175,443,38285,1656.999352,0.000001,2,...,0.000,0.00000,0.0,0.0,0.0,0.00,0.0,0.0,NeedManualLabel,True
4,184,3c:18:a0:41:c3:a0,Arlo Q Indoor Camera,54.154.34.86,192.168.137.175,443,35130,383.612209,0.000726,2,...,0.000,0.00000,0.0,0.0,0.0,0.00,0.0,0.0,NeedManualLabel,True


In [76]:
# Convert timestamps to explicit datetime objects for reliable sorting
combined_features['flow_Timestamp'] = pd.to_datetime(combined_features['flow_Timestamp'])

# Cast IP columns to strings to ensure consistent formatting
ip1 = combined_features['flow_Src IP'].astype(str)
ip2 = combined_features['flow_Dst IP'].astype(str)

# Vectorized sorting of IPs to create bi directional conversation keys
combined_features['bi_dir_key'] = np.where(
    ip1 < ip2, 
    ip1 + " <-> " + ip2, 
    ip2 + " <-> " + ip1
)

print(f"Generated {combined_features['bi_dir_key'].nunique()} unique bi-directional conversation keys.")

/var/folders/x1/6wvrwr515414grvsnzqx0lv40000gn/T/ipykernel_96642/3397800402.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  combined_features['flow_Timestamp'] = pd.to_datetime(combined_features['flow_Timestamp'])


Generated 1635 unique bi-directional conversation keys.


In [77]:
conv_packet_counts = combined_features.groupby('bi_dir_key').size().sort_values(ascending=False)

print(f"Total Unique Bi-directional Groups: {len(conv_packet_counts)}")
print("\nTop 20 Largest Conversations (by row/packet count):")
print(conv_packet_counts.head(20))

Total Unique Bi-directional Groups: 1635

Top 20 Largest Conversations (by row/packet count):
bi_dir_key
192.168.137.46 <-> 35.185.101.66      25648
157.249.81.141 <-> 192.168.137.41     12002
192.168.137.172 <-> 192.168.137.58     9736
192.168.137.175 <-> 63.32.36.73        2766
192.168.137.175 <-> 99.81.57.97        2734
192.168.137.1 <-> 192.168.137.41       2691
192.168.137.175 <-> 54.72.111.213      2596
192.168.137.175 <-> 3.248.67.251       2501
192.168.137.175 <-> 63.35.30.30        2352
192.168.137.175 <-> 52.31.220.16       2248
192.168.137.175 <-> 34.246.132.214     1994
192.168.137.175 <-> 52.214.197.222     1667
192.168.137.58 <-> 192.168.137.83      1480
192.168.137.175 <-> 54.77.42.87        1476
192.168.137.35 <-> 51.145.143.28       1465
192.168.137.128 <-> 192.168.137.61     1461
192.168.137.175 <-> 34.243.231.175     1330
192.168.137.175 <-> 52.211.91.196      1328
192.168.137.175 <-> 54.154.34.86       1304
192.168.137.175 <-> 34.241.186.42      1273
dtype: int64


In [78]:
# See the statistical breakdown of your group sizes
print(conv_packet_counts.describe())

count     1635.000000
mean       124.890520
std        784.661489
min          1.000000
25%          2.000000
50%          6.000000
75%         34.000000
max      25648.000000
dtype: float64


In [79]:
# Bin the groups to see how many conversations are large vs small
bins = [0, 5, 20, 100, 1000, np.inf]
labels = ['1-5 rows', '6-20 rows', '21-100 rows', '101-1000 rows', '1000+ rows']
distribution = pd.cut(conv_packet_counts, bins=bins, labels=labels).value_counts()

print("Distribution of Conversation Sizes:")
print(distribution)

Distribution of Conversation Sizes:
1-5 rows         796
6-20 rows        347
101-1000 rows    237
21-100 rows      224
1000+ rows        31
Name: count, dtype: int64


In [80]:
# Isolate the specific conversation
target_group = combined_features[combined_features['bi_dir_key'] == '192.168.137.175 <-> 54.72.111.213']

# See the total rows and columns to confirm
print(target_group.shape)

(2596, 222)


In [81]:
sort_by = [
    "flow_Timestamp", #Coarse chronological order (seconds)
    "pkt_time_since_previously_displayed_frame", #Fine-grained packet timing within the same second
    "pkt_inter_arrival_time", #Additional timing resolution
    "pkt_stream", #Keep packets from the same conversation together
]

# Columns to inspect
sample_cols = [
    "flow_Src IP",
    "flow_Dst IP",
    "pkt_stream_1_mean",
    "pkt_label",
    *sort_by,
]

# Remove duplicates while preserving order
sample_cols = list(dict.fromkeys(sample_cols))

# Sort and inspect
target_group[sample_cols].sort_values(
        by=sort_by,
        ascending=True,
    ).head(100000)

,flow_Src IP,flow_Dst IP,pkt_stream_1_mean,pkt_label,flow_Timestamp,pkt_time_since_previously_displayed_frame,pkt_inter_arrival_time,pkt_stream
81260,192.168.137.175,54.72.111.213,572.565217,Benign,2022-07-10 23:38:35,0.000000,490.990510,270
153045,192.168.137.175,54.72.111.213,1491.390244,Benign,2022-07-10 23:38:35,0.000000,495.937170,270
8160,192.168.137.175,54.72.111.213,1128.775000,Benign,2022-07-10 23:38:35,0.000000,502.681800,270
19521,192.168.137.175,54.72.111.213,1402.112500,Benign,2022-07-10 23:38:35,0.000001,488.428196,270
185492,192.168.137.175,54.72.111.213,1523.600000,Benign,2022-07-10 23:38:35,0.000001,488.545013,270
...,...,...,...,...,...,...,...,...
195706,192.168.137.175,54.72.111.213,786.756098,Benign,2022-08-10 00:08:28,0.081294,2403.593974,973
185876,192.168.137.175,54.72.111.213,1314.500000,Benign,2022-08-10 00:08:28,0.084442,2391.685994,973
25200,192.168.137.175,54.72.111.213,1085.526316,Benign,2022-08-10 00:08:28,0.099410,2332.574363,973
83065,192.168.137.175,54.72.111.213,917.400000,Benign,2022-08-10 00:08:28,0.106542,2429.268664,973


In [82]:
# Create the label distribution matrix
label_matrix = pd.crosstab(combined_features['bi_dir_key'], combined_features['pkt_label'])

# Sort by the total number of packets so the largest conversations are at the top
label_matrix['Total'] = label_matrix.sum(axis=1)
label_matrix = label_matrix.sort_values(by='Total', ascending=False)

label_matrix.head(10000)

pkt_label,Benign,Brute Force,DDoS-HTTP Flood,DNS Spoofing,DoS-HTTP Flood,XSS,Total
bi_dir_key,,,,,,,
192.168.137.46 <-> 35.185.101.66,25648,0,0,0,0,0,25648
157.249.81.141 <-> 192.168.137.41,11999,0,3,0,0,0,12002
192.168.137.172 <-> 192.168.137.58,9736,0,0,0,0,0,9736
192.168.137.175 <-> 63.32.36.73,2766,0,0,0,0,0,2766
192.168.137.175 <-> 99.81.57.97,2734,0,0,0,0,0,2734
...,...,...,...,...,...,...,...
192.168.137.28 <-> 47.254.14.172,0,0,0,1,0,0,1
192.168.137.254 <-> 54.164.163.115,0,0,0,0,0,1,1
192.168.137.108 <-> 239.255.255.250,0,0,0,0,0,1,1


In [130]:
# Window: This would set the amount of conversations at once that the model would train on
conv_window = 5

exclude = {
    "pkt_label",
    "flow_Label",
    "flow_matched",
    "bi_dir_key",
    "time_since_previously_displayed_frame",
    "src_ip",
    "dst_ip",
    "src_port",
    "dst_port",
    "flow_Timestamp",

    "pkt_src_mac",
    "pkt_dst_mac",
    "pkt_tls_server",
    "pkt_http_request_method",
    "pkt_http_request_method",
    "pkt_http_host",
    "pkt_user_agent",
    "pkt_dns_server",
    "pkt_device_mac",
    "pkt_eth_src_oui",
    "pkt_eth_dst_oui",
    "pkt_highest_layer", #UDP, TCP etc... i'll take a look later
    "pkt_http_uri",
    "pkt_http_content_type",
    
}

# I would differentiate the packet and flow data from here by labels
combined_allowed_labels = [
    c for c in combined_features
        .select_dtypes(include=[np.number])
        .columns
    if c not in exclude
]
flow_allowed_labels = [
    c for c in flow_features
        .select_dtypes(include=[np.number])
        .columns
    if c not in exclude
]

In [131]:
# Replace infinities with NaN
combined_features[combined_allowed_labels] = (
    combined_features[combined_allowed_labels]
        .replace([np.inf, -np.inf], np.nan)
)

flow_features[flow_allowed_labels] = (
    flow_features[flow_allowed_labels]
        .replace([np.inf, -np.inf], np.nan)
)

# Fill NaNs with the median of each column
combined_features[combined_allowed_labels] = (
    combined_features[combined_allowed_labels]
        .fillna(
            combined_features[combined_allowed_labels].median()
        )
)

flow_features[flow_allowed_labels] = (
    flow_features[flow_allowed_labels]
        .fillna(
            flow_features[flow_allowed_labels].median()
        )
)

# Force float64
combined_features[combined_allowed_labels] = (
    combined_features[combined_allowed_labels]
        .astype(np.float64)
)

flow_features[flow_allowed_labels] = (
    flow_features[flow_allowed_labels]
        .astype(np.float64)
)

In [132]:
X_model1 = []
y_model1 = []

In [133]:
# Model 1 data generation
for _, group in combined_features.groupby("bi_dir_key"):

    group = group.sort_values(sort_by).reset_index(drop=True)

    # Drop conversations smaller than the window
    if len(group) < conv_window:
        continue

    # Non-overlapping windows
    # for start in range(0, len(group), conv_window): # [1,2,3,4, 5], [6, 7, 8, 9, 10]
    for start in range(len(group) - conv_window + 1): # [1,2,3,4,5], [2,3,4,5,6]

        window = group.iloc[start:start + conv_window]

        if len(window) != conv_window:
            continue

        X_model1.append(window[combined_allowed_labels].to_numpy())

        labels = window["pkt_label"].astype(str)

        malicious = labels[labels.str.lower() != "benign"]

        if len(malicious):
            y_model1.append(malicious.iloc[0])
        else:
            y_model1.append("benign")

In [134]:
X_model1 = np.array(X_model1)
y_model1 = np.array(y_model1)

In [57]:
len(y_model1) #before sliding window

80480

In [135]:
len(y_model1) #after sliding window

199150

In [136]:
label_distribution = (
    pd.Series(y_model1)
    .value_counts()
    .rename_axis("Label")
    .reset_index(name="Count")
)

label_distribution["Percentage"] = (
    label_distribution["Count"] / label_distribution["Count"].sum() * 100
).round(2)

print(label_distribution)

             Label   Count  Percentage
0           benign  196652       98.75
1      Brute Force    1418        0.71
2     DNS Spoofing     455        0.23
3              XSS     443        0.22
4  DDoS-HTTP Flood     146        0.07
5   DoS-HTTP Flood      36        0.02


In [137]:
# Data ready... time to train

In [138]:
X_model1.shape

(199150, 5, 198)

In [139]:
#Separate benign data

benign_mask = y_model1 == "benign"

X_benign = X_model1[benign_mask]

print(X_benign.shape)

(196652, 5, 198)


In [140]:
X_train, X_val = train_test_split(
    X_benign,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

In [141]:
#Normalize data

samples, timesteps, features = X_train.shape

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train.reshape(-1, features)
).reshape(samples, timesteps, features)

X_val_scaled = scaler.transform(
    X_val.reshape(-1, features)
).reshape(X_val.shape)

In [143]:
X_train_flat = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    -1
)

X_val_flat = X_val_scaled.reshape(
    X_val_scaled.shape[0],
    -1
)

input_dim = X_train_flat.shape[1]

print(input_dim)

990


In [173]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

X_train_tensor = torch.tensor(
    X_train_flat,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val_flat,
    dtype=torch.float32
)

train_loader = DataLoader(
    TensorDataset(X_train_tensor),
    batch_size=256,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val_tensor),
    batch_size=256,
    shuffle=False
)

In [174]:
class PacketAutoencoder(nn.Module):

    def __init__(self, input_dim, latent_dim=32):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),

            nn.Linear(256,128),
            nn.ReLU(),

            nn.Linear(128,64),
            nn.ReLU(),

            nn.Linear(64,latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim,64),
            nn.ReLU(),

            nn.Linear(64,128),
            nn.ReLU(),

            nn.Linear(128,256),
            nn.ReLU(),

            nn.Linear(256,input_dim)
        )

    def forward(self,x):

        latent = self.encoder(x)

        reconstruction = self.decoder(latent)

        return reconstruction, latent

In [175]:
model = PacketAutoencoder(input_dim).to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [176]:
epochs = 50

train_losses = []
val_losses = []

for epoch in range(epochs):

    ###################
    # Train
    ###################

    model.train()

    running_loss = 0

    for (batch,) in train_loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        reconstruction, latent = model(batch)

        loss = criterion(reconstruction, batch)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * batch.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    ###################
    # Validation
    ###################

    model.eval()

    running_loss = 0

    with torch.no_grad():

        for (batch,) in val_loader:

            batch = batch.to(device)

            reconstruction, latent = model(batch)

            loss = criterion(reconstruction, batch)

            running_loss += loss.item() * batch.size(0)

    val_loss = running_loss / len(val_loader.dataset)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch {epoch+1:3d} | "
        f"Train {train_loss:.6f} | "
        f"Val {val_loss:.6f}"
    )

Epoch   1 | Train 0.336732 | Val 0.228735
Epoch   2 | Train 0.197634 | Val 0.180843
Epoch   3 | Train 0.162621 | Val 0.154492
Epoch   4 | Train 0.140493 | Val 0.136326
Epoch   5 | Train 0.124676 | Val 0.125719
Epoch   6 | Train 0.113997 | Val 0.113500
Epoch   7 | Train 0.105713 | Val 0.111116
Epoch   8 | Train 0.101456 | Val 0.103195
Epoch   9 | Train 0.095661 | Val 0.098431
Epoch  10 | Train 0.092634 | Val 0.095962
Epoch  11 | Train 0.089713 | Val 0.100246
Epoch  12 | Train 0.087023 | Val 0.098592
Epoch  13 | Train 0.083550 | Val 0.087323
Epoch  14 | Train 0.080322 | Val 0.093027
Epoch  15 | Train 0.079955 | Val 0.084810
Epoch  16 | Train 0.079451 | Val 0.084690
Epoch  17 | Train 0.076362 | Val 0.083069
Epoch  18 | Train 0.075127 | Val 0.123162
Epoch  19 | Train 0.078776 | Val 0.078532
Epoch  20 | Train 0.076892 | Val 0.076270
Epoch  21 | Train 0.070509 | Val 0.085823
Epoch  22 | Train 0.072101 | Val 0.077635
Epoch  23 | Train 0.072693 | Val 0.084398
Epoch  24 | Train 0.070875 | Val 0

In [178]:
model.eval()

errors = []

with torch.no_grad():

    for (batch,) in val_loader:

        batch = batch.to(device)

        reconstruction, _ = model(batch)

        mse = torch.mean(
            (batch - reconstruction) ** 2,
            dim=1
        )

        errors.extend(
            mse.cpu().numpy()
        )

errors = np.array(errors)

threshold = np.percentile(errors, 99)

print("Threshold:", threshold)

Threshold: 0.37459472


In [179]:
X_all = scaler.transform(
    X_model1.reshape(-1, features)
).reshape(X_model1.shape)

X_all = X_all.reshape(X_all.shape[0], -1)

X_all = torch.tensor(
    X_all,
    dtype=torch.float32
)

loader = DataLoader(
    TensorDataset(X_all),
    batch_size=512,
    shuffle=False
)

In [180]:
model.eval()

scores = []

with torch.no_grad():

    for (batch,) in loader:

        batch = batch.to(device)

        reconstruction, _ = model(batch)

        mse = torch.mean(
            (batch - reconstruction) ** 2,
            dim=1
        )

        scores.extend(
            mse.cpu().numpy()
        )

scores = np.array(scores)

In [181]:
predicted = np.where(
    scores > threshold,
    "anomaly",
    "benign"
)

actual = np.where(
    y_model1 == "benign",
    "benign",
    "anomaly"
)

In [182]:
print(classification_report(actual, predicted))

              precision    recall  f1-score   support

     anomaly       0.55      0.77      0.64      2498
      benign       1.00      0.99      0.99    196652

    accuracy                           0.99    199150
   macro avg       0.77      0.88      0.82    199150
weighted avg       0.99      0.99      0.99    199150



In [183]:
# Binary ground truth
y_test = (y_model1 != "benign").astype(int)

# Binary predictions from reconstruction error
y_pred = (scores > threshold).astype(int)

# -----------------------------
# AUC-ROC
# -----------------------------
auc = roc_auc_score(y_test, scores)

print(f"Autoencoder Reconstruction Error AUC-ROC Score: {auc:.4f}")

# -----------------------------
# Confusion Matrix
# -----------------------------
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(f"True Benign (Predicted Benign):      {cm[0,0]}")
print(f"False Positive (False Alarm):        {cm[0,1]}")
print(f"False Negative (Missed Detection):   {cm[1,0]}")
print(f"True Anomaly (Correct Detection):    {cm[1,1]}")

# -----------------------------
# Classification Report
# -----------------------------
print("\nClassification Report")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Benign", "Anomaly"],
    )
)

Autoencoder Reconstruction Error AUC-ROC Score: 0.9827

Confusion Matrix:
True Benign (Predicted Benign):      195075
False Positive (False Alarm):        1577
False Negative (Missed Detection):   580
True Anomaly (Correct Detection):    1918

Classification Report
              precision    recall  f1-score   support

      Benign       1.00      0.99      0.99    196652
     Anomaly       0.55      0.77      0.64      2498

    accuracy                           0.99    199150
   macro avg       0.77      0.88      0.82    199150
weighted avg       0.99      0.99      0.99    199150



In [160]:
# print(f"Precision : {precision_score(y_test, y_pred):.4f}")
# print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
# print(f"F1-Score  : {f1_score(y_test, y_pred):.4f}")